# Local Brain: Gemma E2B Fine-Tuning with Unsloth

This notebook fine-tunes **Google's Gemma E2B** on your custom enterprise dataset.

- **Runtime:** Free Google Colab T4 GPU (~15 GB VRAM)
- **Training Time:** ~10–12 minutes
- **Output:** Quantized `q4_k_m.gguf` file (~1.3 GB) ready to run on your local **NVIDIA GTX 1050 Ti (4GB)** via `node-llama-cpp` completely offline!

## 1. Install Unsloth & Core Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load Base Gemma Model with Unsloth

In [ ]:
import os
import torch
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_SEQ_LENGTH = 2048
HF_TOKEN = os.environ.get("HF_TOKEN", None)  # Set or use Colab secrets userdata.get("HF_TOKEN")

MODEL_CANDIDATES = [
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-2-2b-it",
]

model = None
tokenizer = None
chosen_name = None

for name in MODEL_CANDIDATES:
    try:
        print(f"[*] Attempting to load: {name}...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=name,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
            token=HF_TOKEN,
        )
        chosen_name = name
        print(f"[✔] Loaded {chosen_name} successfully!")
        break
    except Exception as e:
        print(f"[!] Notice: {name} not available or gated: {e}")

if model is None:
    raise RuntimeError("Could not load any Gemma model candidate.")

# Set chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma")

## 3. Attach LoRA PEFT Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("[✔] LoRA configured with fast Unsloth kernels.")

## 4. Upload Training Dataset (`train.jsonl` and `eval.jsonl`)

Run this cell to upload your local `data/train.jsonl` and `data/eval.jsonl` from your machine.

In [ ]:
from google.colab import files
import json

if not os.path.exists("train.jsonl"):
    print("Please select and upload 'train.jsonl' and 'eval.jsonl' from your local-brain/data/ directory:")
    uploaded = files.upload()

def load_dataset_file(filename):
    records = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    print(f"Loaded {len(records)} records from {filename}")
    return records

def prepare_gemma_dataset(records, tok):
    formatted = []
    for r in records:
        reformatted = []
        system_prefix = ""
        for m in r["messages"]:
            if m["role"] == "system":
                system_prefix += m["content"] + "\n\n"
            elif m["role"] == "user":
                reformatted.append({"role": "user", "content": (system_prefix + m["content"]).strip()})
                system_prefix = ""
            elif m["role"] == "assistant":
                reformatted.append({"role": "model", "content": m["content"]})
        text = tok.apply_chat_template(reformatted, tokenize=False, add_generation_prompt=False)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

train_records = load_dataset_file("train.jsonl")
eval_records = load_dataset_file("eval.jsonl")

train_dataset = prepare_gemma_dataset(train_records, tokenizer)
eval_dataset = prepare_gemma_dataset(eval_records, tokenizer)
print("[✔] Datasets successfully mapped to Gemma chat format!")

## 5. Train with Response-Only Masking

Masks instruction tokens so the model only calculates loss on model outputs.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="no",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("[*] Starting training...")
stats = trainer.train()
print("[✔] Training finished successfully!")

## 6. Export to GGUF (Q4_K_M) and Download

This compiles the model directly into a `q4_k_m.gguf` binary ready to drop into `models/llm/` on your GTX 1050 Ti.

In [ ]:
EXPORT_PATH = "local_brain_gemma_e2b"

print("[*] Exporting to Q4_K_M GGUF...")
model.save_pretrained_gguf(
    EXPORT_PATH,
    tokenizer,
    quantization_method="q4_k_m",
)
print("[✔] Export complete!")

# Find the generated GGUF file and trigger browser download
import glob
gguf_files = glob.glob(f"{EXPORT_PATH}/*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    print(f"Downloading {target_file} to your machine...")
    files.download(target_file)
else:
    print("Could not locate exported .gguf file. Check directory.")